# 06-2. TCP 클라이언트 예제

## Goal

- `recv()` 한 번이 전체 메시지를 보장하지 않는 이유를 확인합니다.
- 연결 종료를 불완전 수신과 구분합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

가짜 소켓으로 부분 수신을 재현하므로 포트를 열지 않습니다.


## Steps

### 필요한 길이만큼 반복 수신

수신 조각이 작더라도 목표 바이트 수가 모일 때까지 반복합니다.


In [1]:
class ScriptedSocket:
    def __init__(self, chunks):
        self.chunks = list(chunks)

    def recv(self, size):
        if not self.chunks:
            return b""
        chunk = self.chunks.pop(0)
        if len(chunk) > size:
            self.chunks.insert(0, chunk[size:])
            return chunk[:size]
        return chunk


def receive_exactly(sock, size: int) -> bytes:
    if size < 0:
        raise ValueError("size는 0 이상이어야 합니다")
    result = bytearray()
    while len(result) < size:
        chunk = sock.recv(size - len(result))
        if not chunk:
            raise ConnectionError("메시지 수신 전에 연결이 종료되었습니다")
        result.extend(chunk)
    return bytes(result)


client_socket = ScriptedSocket([b"HE", b"L", b"LO"])
message = receive_exactly(client_socket, 5)
print(message, message.decode("ascii"))


b'HELLO' HELLO


## Checks

부분 수신과 조기 종료를 각각 확인합니다.


In [2]:
assert message == b"HELLO"
try:
    receive_exactly(ScriptedSocket([b"AB"]), 3)
except ConnectionError as error:
    print("예상한 오류:", error)
else:
    raise AssertionError("조기 종료를 발견하지 못했습니다")


예상한 오류: 메시지 수신 전에 연결이 종료되었습니다


## Next Steps

실제 클라이언트에서는 연결 타임아웃과 읽기 타임아웃을 모두 설정합니다.
